> **Note:** This notebook requires Apache Spark + Delta Lake (`pyspark`, `delta-spark`) with internet access to download the Delta JAR from Maven Central. It is designed to run on **Databricks** (Delta is built in there — no setup needed), or on any local machine with Spark configured for Delta Lake. See `README.md` in this repo for exact setup steps.

# Delta Lake MERGE Implementation

**Objective:** Learn Delta Lake / Spark basics and perform data exploration,
cleaning, and MERGE (upsert) operations using the Superstore dataset.

**Steps (as specified):**
1. Load a CSV dataset into a Spark DataFrame
2. Explore data (head/tail, shape, columns, data types)
3. Handle missing values (identify, fill/drop)
4. Perform basic operations (filter rows, select columns)
5. Remove duplicates
6. Create a derived column (`total_amount = price * quantity`)
7. Save the cleaned dataset as a new Delta table + CSV file
8. **Delta Lake MERGE Implementation** — upserts using `MERGE INTO`, with ACID/versioning proof

**Dataset:** `Sample - Superstore.csv`
(Source: https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

**Reference:** https://learn.microsoft.com/en-us/azure/databricks/delta/merge

**Output:** Notebook (.ipynb) + cleaned CSV + brief summary (this notebook produces all three)

## Setup
Update `csv_path` below to match where you uploaded `Sample - Superstore.csv`
(Volume / DBFS path).

In [0]:
csv_path = "/Volumes/workspace/default/superstore_data/archive (6)/Sample - Superstore.csv"

spark.sql("CREATE DATABASE IF NOT EXISTS superstore_delta")
spark.sql("USE superstore_delta")
print("Using database: superstore_delta")

Using database: superstore_delta


## 1. Load a CSV dataset into a Spark DataFrame
(Delta Lake equivalent of loading a CSV into a Pandas DataFrame — here we load
it into a Spark DataFrame, which we will then persist as a Delta table.)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(csv_path)
)

# Clean up column names (remove spaces/special chars so they work well in SQL)
for c in df.columns:
    new_c = c.strip().replace(" ", "_").replace("-", "_")
    df = df.withColumnRenamed(c, new_c)

# Ensure numeric columns have the correct data type (CSV can read them as strings)
df = (df
    .withColumn("Sales", F.col("Sales").cast("double"))
    .withColumn("Quantity", F.col("Quantity").cast("int"))
    .withColumn("Discount", F.col("Discount").cast("double"))
    .withColumn("Profit", F.col("Profit").cast("double")))

print("Dataset loaded successfully!")
df.show(5)

Dataset loaded successfully!
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|  Segment|      Country|           City|     State|Postal_Code|Region|     Product_ID|       Category|Sub_Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furnitur

## 2. Explore data (head/tail, shape, columns, data types)

In [0]:
# Head (first 5 rows)
print("--- Head (first 5 rows) ---")
df.show(5)

--- Head (first 5 rows) ---
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|  Segment|      Country|           City|     State|Postal_Code|Region|     Product_ID|       Category|Sub_Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture

In [0]:
# Tail (last 5 rows) — Spark has no native tail(), so we order by Row_ID descending
print("--- Tail (last 5 rows) ---")
df.orderBy(F.col("Row_ID").desc()).show(5)

--- Tail (last 5 rows) ---
+------+--------------+----------+----------+--------------+-----------+----------------+--------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|   Customer_Name| Segment|      Country|       City|     State|Postal_Code|Region|     Product_ID|       Category|Sub_Category|        Product_Name|  Sales|Quantity|Discount| Profit|
+------+--------------+----------+----------+--------------+-----------+----------------+--------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|  9994|CA-2017-119914|2017-05-04|2017-05-09|  Second Class|   CC-12220|    Chris Cortes|Consumer|United States|Westminster|California|      92683|  West|OFF-AP-10002684|Office Supplies|  Appliances|Acco 7-Ou

In [0]:
# Shape (rows, columns)
n_rows = df.count()
n_cols = len(df.columns)
print(f"Shape (rows, columns): ({n_rows}, {n_cols})")

Shape (rows, columns): (9994, 21)


In [0]:
# Column names
print("Columns:")
print(df.columns)

Columns:
['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [0]:
# Data types of each column
df.printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
# Quick statistical summary of numeric columns
df.select("Sales", "Quantity", "Discount", "Profit").describe().show()

+-------+-----------------+------------------+-------------------+------------------+
|summary|            Sales|          Quantity|           Discount|            Profit|
+-------+-----------------+------------------+-------------------+------------------+
|  count|             9994|              9994|               9994|              9994|
|   mean|229.8580008304938| 3.789573744246548|0.15620272163298934|28.656896307784802|
| stddev|623.2451005086805|2.2251096911413994| 0.2064519678257168|234.26010769095768|
|    min|            0.444|                 1|                0.0|         -6599.978|
|    max|         22638.48|                14|                0.8|          8399.976|
+-------+-----------------+------------------+-------------------+------------------+



## 3. Handle missing values (identify, fill/drop)

In [0]:
# Identify missing values in each column
missing_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
])
print("Missing values per column:")
missing_counts.show(vertical=True, truncate=False)

total_missing = sum(row[0] for row in missing_counts.collect())
print("Total missing values in dataset:", total_missing)

Missing values per column:
-RECORD 0------------
 Row_ID        | 0   
 Order_ID      | 0   
 Order_Date    | 0   
 Ship_Date     | 0   
 Ship_Mode     | 0   
 Customer_ID   | 0   
 Customer_Name | 0   
 Segment       | 0   
 Country       | 0   
 City          | 0   
 State         | 0   
 Postal_Code   | 0   
 Region        | 0   
 Product_ID    | 0   
 Category      | 0   
 Sub_Category  | 0   
 Product_Name  | 0   
 Sales         | 0   
 Quantity      | 0   
 Discount      | 0   
 Profit        | 0   

Total missing values in dataset: 0


In [0]:
# Demonstration: how you would fill / drop missing values if they existed
# (technique shown on a temporary copy so the original DataFrame stays untouched)
demo = df.withColumn(
    "Sales",
    F.when(F.col("Row_ID") <= 3, F.lit(None).cast("double")).otherwise(F.col("Sales"))
)  # artificially introduce a few missing values

print("Missing values before handling:", demo.filter(F.col("Sales").isNull()).count())

# Option A: Fill missing numeric values with the column mean
mean_sales = demo.select(F.mean("Sales")).collect()[0][0]
demo_filled = demo.fillna({"Sales": mean_sales})

# Option B: Drop rows that still contain missing values
demo_dropped = demo.dropna(subset=["Sales"])

print("Missing values after fillna:", demo_filled.filter(F.col("Sales").isNull()).count())
print("Rows remaining after dropna:", demo_dropped.count(), "out of", demo.count())

Missing values before handling: 3
Missing values after fillna: 0
Rows remaining after dropna: 9991 out of 9994


In [0]:
# Since the real dataset has no missing values, we simply confirm it is clean
df_clean = df
print("Missing values in the working dataset:", total_missing)

Missing values in the working dataset: 0


## 4. Perform basic operations (filter rows, select columns)

In [0]:
# Select specific columns
subset_cols = df_clean.select("Order_ID", "Customer_Name", "Category", "Sales", "Quantity", "Profit")
subset_cols.show(5)

+--------------+---------------+---------------+--------+--------+--------+
|      Order_ID|  Customer_Name|       Category|   Sales|Quantity|  Profit|
+--------------+---------------+---------------+--------+--------+--------+
|CA-2016-152156|    Claire Gute|      Furniture|  261.96|       2| 41.9136|
|CA-2016-152156|    Claire Gute|      Furniture|  731.94|       3| 219.582|
|CA-2016-138688|Darrin Van Huff|Office Supplies|   14.62|       2|  6.8714|
|US-2015-108966| Sean O'Donnell|      Furniture|957.5775|       5|-383.031|
|US-2015-108966| Sean O'Donnell|Office Supplies|  22.368|       2|  2.5164|
+--------------+---------------+---------------+--------+--------+--------+
only showing top 5 rows


In [0]:
# Filter rows: orders from the 'Technology' category with Profit > 100
tech_profitable = df_clean.filter((F.col("Category") == "Technology") & (F.col("Profit") > 100))
print("Number of profitable Technology orders:", tech_profitable.count())
tech_profitable.show(5)

Number of profitable Technology orders: 354
+------+--------------+----------+----------+--------------+-----------+--------------+-----------+-------------+-------------+---------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID| Customer_Name|    Segment|      Country|         City|    State|Postal_Code| Region|     Product_ID|  Category|Sub_Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+--------------+-----------+-------------+-------------+---------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+--------+
|    36|CA-2016-117590|2016-12-08|2016-12-10|   First Class|   GH-14485|     Gene Hale|  Corporate|United States|   Richardson|    Texas|      75080|Central|TEC-PH-10004977|Technology|      P

In [0]:
# Filter rows: orders shipped via 'Same Day'
same_day = df_clean.filter(F.col("Ship_Mode") == "Same Day")
print("Number of Same Day shipped orders:", same_day.count())

Number of Same Day shipped orders: 543


## 5. Remove duplicates

In [0]:
dup_count = df_clean.count() - df_clean.dropDuplicates().count()
print("Number of duplicate rows before removal:", dup_count)

df_clean = df_clean.dropDuplicates()

print("Shape after removing duplicates:", (df_clean.count(), len(df_clean.columns)))

Number of duplicate rows before removal: 0
Shape after removing duplicates: (9994, 21)


## 6. Create a derived column (`total_amount = price * quantity`)
In this dataset, `Sales` already represents the total sale amount for the
line item. To follow the required formula (`price * quantity`), we first
derive a `Price` (unit price) column as `Sales / Quantity`, then compute
`total_amount = Price * Quantity` (which naturally reconciles back to `Sales`).

In [0]:
df_clean = (df_clean
    .withColumn("Price", F.round(F.col("Sales") / F.col("Quantity"), 2))
    .withColumn("total_amount", F.round(F.col("Price") * F.col("Quantity"), 2))
)

df_clean.select("Sales", "Quantity", "Price", "total_amount").show(5)

+-------+--------+-----+------------+
|  Sales|Quantity|Price|total_amount|
+-------+--------+-----+------------+
|   8.56|       2| 4.28|        8.56|
|  21.39|       1|21.39|       21.39|
| 102.36|       3|34.12|      102.36|
| 74.112|       8| 9.26|       74.08|
|130.464|       6|21.74|      130.44|
+-------+--------+-----+------------+
only showing top 5 rows


## 7. Save the cleaned dataset as a new Delta table + CSV file
We save the cleaned dataset in two forms:
- As a **Delta table** (`superstore_delta.cleaned_orders`) — gives us ACID
writes and versioning for free
- As a **CSV file** — for the "cleaned CSV" deliverable required by the
assignment output

In [0]:
(df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("superstore_delta.cleaned_orders"))

print("Cleaned dataset saved as Delta table: superstore_delta.cleaned_orders")

Cleaned dataset saved as Delta table: superstore_delta.cleaned_orders


In [0]:
# Also export as a single CSV file (for the required "cleaned CSV" output)
output_csv_path = "/Volumes/workspace/default/superstore_data/Superstore_cleaned"

(df_clean.coalesce(1).write
    .format("csv")
    .option("header", True)
    .mode("overwrite")
    .save(output_csv_path))

print(f"Cleaned dataset also saved as CSV at: {output_csv_path}")

Cleaned dataset also saved as CSV at: /Volumes/workspace/default/superstore_data/Superstore_cleaned


## 8. Delta Lake MERGE Implementation — Upserts, ACID & Versioning
This is the core "Delta Lake MERGE Implementation" part of the assignment:
using `MERGE INTO` to perform **upserts** (update matched rows, insert new
ones) on top of the cleaned Delta table, and proving Delta Lake's ACID +
versioning guarantees.

### 8a. ACID + Versioning — inspecting the Delta transaction log
Every write to a Delta table creates a new **version**, tracked in the
`_delta_log`. `DESCRIBE HISTORY` shows the full audit trail, and time
travel lets us query any earlier version.

```sql
DESCRIBE HISTORY superstore_delta.cleaned_orders

### 8b. Simulate an incoming batch (new data to upsert)
In a real pipeline, new data (corrected orders, brand-new orders) arrives
daily. We simulate that here with a small incremental DataFrame: a mix of
**corrections** to existing order lines and **brand-new** order lines.

In [0]:
existing_rows = (spark.table("superstore_delta.cleaned_orders")
                  .select("Row_ID", "Order_ID", "Product_ID")
                  .limit(2)
                  .collect())

updates_data = [
    # Corrections to existing order lines (Quantity & Discount changed)
    (existing_rows[0].Row_ID, existing_rows[0].Order_ID, existing_rows[0].Product_ID, 5, 0.10, 55.00, 25.00, 125.00),
    (existing_rows[1].Row_ID, existing_rows[1].Order_ID, existing_rows[1].Product_ID, 4, 0.00, 30.00, 40.00, 160.00),
    # New order lines
    (999901, "CA-2026-000001", existing_rows[0].Product_ID, 3, 0.00, 45.00, 60.00, 180.00),
    (999902, "CA-2026-000002", existing_rows[1].Product_ID, 2, 0.20, 15.00, 35.00, 70.00),
]
updates_df = spark.createDataFrame(
    updates_data,
    ["Row_ID", "Order_ID", "Product_ID", "Quantity", "Discount", "Profit", "Price", "total_amount"]
)
display(updates_df)

Row_ID,Order_ID,Product_ID,Quantity,Discount,Profit,Price,total_amount
19,CA-2014-143336,OFF-AR-10003056,5,0.1,55.0,25.0,125.0
83,CA-2014-139451,OFF-ST-10002370,4,0.0,30.0,40.0,160.0
999901,CA-2026-000001,OFF-AR-10003056,3,0.0,45.0,60.0,180.0
999902,CA-2026-000002,OFF-ST-10002370,2,0.2,15.0,35.0,70.0


### 8c. Upsert using `MERGE INTO`
Matched order lines (same `Row_ID`) get their measures corrected; unmatched
(new) order lines get inserted — all in a single atomic operation.

In [0]:
from delta.tables import DeltaTable

cleaned_dt = DeltaTable.forName(spark, "superstore_delta.cleaned_orders")

(cleaned_dt.alias("target")
    .merge(
        updates_df.alias("source"),
        "target.Row_ID = source.Row_ID"
    )
    .whenMatchedUpdate(set={
        "Order_ID": "source.Order_ID",
        "Product_ID": "source.Product_ID",
        "Quantity": "source.Quantity",
        "Discount": "source.Discount",
        "Profit": "source.Profit",
        "Price": "source.Price",
        "total_amount": "source.total_amount",
    })
    .whenNotMatchedInsert(values={
        "Row_ID": "source.Row_ID",
        "Order_ID": "source.Order_ID",
        "Product_ID": "source.Product_ID",
        "Quantity": "source.Quantity",
        "Discount": "source.Discount",
        "Profit": "source.Profit",
        "Price": "source.Price",
        "total_amount": "source.total_amount",
        # Columns not present in source — set to NULL for new rows
        "Order_Date": "NULL",
        "Ship_Date": "NULL",
        "Ship_Mode": "NULL",
        "Customer_ID": "NULL",
    })
    .execute())

print("Row count after MERGE:", spark.table("superstore_delta.cleaned_orders").count())
display(spark.table("superstore_delta.cleaned_orders").filter(
    F.col("Row_ID").isin([r.Row_ID for r in existing_rows] + [999901, 999902])
))

Row count after MERGE: 9996


Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit,Price,total_amount
83,CA-2014-139451,2014-10-12,2014-10-16,Standard Class,DN-13690,Duane Noonan,Consumer,United States,San Francisco,California,94122,West,OFF-ST-10002370,Office Supplies,Storage,"Sortfiler Multipurpose Personal File Organizer, Black",21.39,4,0.0,30.0,40.0,160.0
19,CA-2014-143336,2014-08-27,2014-09-01,Second Class,ZD-21925,Zuschuss Donatelli,Consumer,United States,San Francisco,California,94109,West,OFF-AR-10003056,Office Supplies,Art,Newell 341,8.56,5,0.1,55.0,25.0,125.0
999902,CA-2026-000002,null,null,null,null,null,null,null,null,null,null,null,OFF-ST-10002370,null,null,null,null,2,0.2,15.0,35.0,70.0
999901,CA-2026-000001,null,null,null,null,null,null,null,null,null,null,null,OFF-AR-10003056,null,null,null,null,3,0.0,45.0,60.0,180.0


### 8d. Verifying ACID behavior after the MERGE
The `MERGE INTO` created a new Delta table version. We inspect the history
again and use time travel to prove the previous version is still available.

```sql
DESCRIBE HISTORY superstore_delta.cleaned_orders

In [0]:
latest_version = spark.sql("DESCRIBE HISTORY superstore_delta.cleaned_orders").agg(F.max("version")).collect()[0][0]
before_merge = spark.read.format("delta").option("versionAsOf", latest_version - 1).table("superstore_delta.cleaned_orders")
after_merge = spark.table("superstore_delta.cleaned_orders")

print("Row count BEFORE merge (version", latest_version - 1, "):", before_merge.count())
print("Row count AFTER merge  (version", latest_version, "):", after_merge.count())

Row count BEFORE merge (version 1 ): 9994
Row count AFTER merge  (version 2 ): 9996


## Brief Summary

- **Step 1-2 (Load & Explore):** Loaded `Sample - Superstore.csv` (9,994 rows,
21 columns) into a Spark DataFrame; inspected head/tail, shape, columns,
and data types.
- **Step 3 (Missing values):** No missing values were found in the dataset;
the fill/drop technique was demonstrated on a temporary copy.
- **Step 4 (Basic operations):** Selected key columns and filtered rows
(e.g. profitable Technology orders, Same Day shipments).
- **Step 5 (Duplicates):** Checked for and removed duplicate rows.
- **Step 6 (Derived column):** Added `total_amount = Price * Quantity`,
where `Price` was derived as `Sales / Quantity`.
- **Step 7 (Save):** Saved the cleaned dataset both as a Delta table
(`superstore_delta.cleaned_orders`) and as a CSV file.
- **Step 8 (Delta Lake MERGE Implementation):** Used `MERGE INTO` to upsert
a simulated incoming batch — corrected existing order lines and inserted
new ones in a single atomic transaction. Verified Delta Lake's ACID
guarantees using `DESCRIBE HISTORY` and time travel (`VERSION AS OF`),
confirming the pre-merge state remained fully recoverable.